# Steam power plant — Rankine cycle analysis & optimisation

**Author:** Esteban López · Industrial Engineer  
**Tools:** Rankine cycle thermodynamics, CoolProp (IAPWS-IF97 steam properties), Python.

## Problem
Analyse a steam power plant operating on a Rankine cycle: compute every state, the thermal efficiency (ideal and
with real turbine/pump efficiencies), the back-work ratio and the turbine-exit quality — then **optimise** the
design by sweeping boiler pressure, superheat temperature and condenser pressure.

**Design point:** boiler 8 MPa, turbine inlet 500 °C, condenser 10 kPa, turbine η = 0.85, pump η = 0.85.

> Reproducible. Steam properties come from **CoolProp** (IAPWS-IF97). `pip install coolprop matplotlib numpy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD, GREY = '#0b1f3a', '#16b3c7', '#f2c14e', '#5b6b7f'
F = 'Water'

## 1. Cycle model
States: 1 = condenser exit (sat. liquid), 2 = pump exit, 3 = turbine inlet (superheated), 4 = turbine exit.

In [ ]:
def rankine(P_boiler, T3, P_cond, eta_t=0.85, eta_p=0.85):
    # 1: saturated liquid at condenser pressure
    h1 = PropsSI('H', 'P', P_cond, 'Q', 0, F); s1 = PropsSI('S', 'P', P_cond, 'Q', 0, F)
    # 2: pump to boiler pressure (isentropic then efficiency)
    h2s = PropsSI('H', 'P', P_boiler, 'S', s1, F)
    wp = (h2s - h1) / eta_p; h2 = h1 + wp
    # 3: superheated steam at boiler pressure
    h3 = PropsSI('H', 'P', P_boiler, 'T', T3, F); s3 = PropsSI('S', 'P', P_boiler, 'T', T3, F)
    # 4: expansion to condenser pressure
    h4s = PropsSI('H', 'P', P_cond, 'S', s3, F)
    wt = eta_t * (h3 - h4s); h4 = h3 - wt
    qin = h3 - h2; wnet = wt - wp; eff = wnet / qin
    x4 = PropsSI('Q', 'P', P_cond, 'H', h4, F)
    return dict(eff=eff, wt=wt, wp=wp, wnet=wnet, qin=qin, x4=x4, bwr=wp/wt,
                h=[h1, h2, h3, h4], s3=s3, s1=s1)

P_b, T3, P_c = 8e6, 500 + 273.15, 10e3
real = rankine(P_b, T3, P_c, 0.85, 0.85)
ideal = rankine(P_b, T3, P_c, 1.0, 1.0)
print(f"Real  cycle: eff = {real['eff']*100:.1f} %   wnet = {real['wnet']/1e3:.0f} kJ/kg   x4 = {real['x4']:.3f}   BWR = {real['bwr']*100:.2f} %")
print(f"Ideal cycle: eff = {ideal['eff']*100:.1f} %   wnet = {ideal['wnet']/1e3:.0f} kJ/kg")
print(f"Turbine work {real['wt']/1e3:.0f} kJ/kg | Pump work {real['wp']/1e3:.1f} kJ/kg | q_in {real['qin']/1e3:.0f} kJ/kg")

## 2. T–s diagram

In [ ]:
# saturation dome
Tc = PropsSI('Tcrit', F)
Tdome = np.linspace(280, Tc-0.5, 120)
sf = [PropsSI('S','T',T,'Q',0,F) for T in Tdome]
sg = [PropsSI('S','T',T,'Q',1,F) for T in Tdome]

h1,h2,h3,h4 = real['h']
s1 = real['s1']; s3 = real['s3']
Tsat_b = PropsSI('T','P',P_b,'Q',0,F)
sf_b = PropsSI('S','P',P_b,'Q',0,F); sg_b = PropsSI('S','P',P_b,'Q',1,F)
x4 = real['x4']; sf_c = PropsSI('S','P',P_c,'Q',0,F); sg_c = PropsSI('S','P',P_c,'Q',1,F)
s4 = sf_c + x4*(sg_c-sf_c); Tc_cond = PropsSI('T','P',P_c,'Q',0,F)
K = 273.15
xs = [s1, s1, sf_b, sg_b, s3, s4, s1]
Ts = [Tc_cond-K, Tc_cond-K, Tsat_b-K, Tsat_b-K, T3-K, Tc_cond-K, Tc_cond-K]
fig, ax = plt.subplots(figsize=(8.5,5.6))
ax.plot(sf, Tdome-K, color=GREY); ax.plot(sg, Tdome-K, color=GREY)
ax.plot(xs, Ts, color=CYAN, lw=2.4)
for s,T,l in [(s1,Tc_cond-K,'1'),(s3,T3-K,'3'),(s4,Tc_cond-K,'4')]:
    ax.plot(s,T,'o',color=GOLD); ax.text(s+0.05,T+6,l,fontweight='bold')
ax.set_xlabel('s (J/kg·K)'); ax.set_ylabel('T (°C)')
ax.set_title('Rankine cycle T–s diagram', fontweight='bold', color=NAVY); plt.tight_layout(); plt.show()

## 3. Optimisation — three levers
Efficiency rises with **boiler pressure**, **superheat temperature** and **lower condenser pressure**.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
Pb = np.linspace(4e6, 18e6, 12)
ax[0].plot(Pb/1e6, [rankine(p, T3, P_c)['eff']*100 for p in Pb], color=CYAN, marker='o')
ax[0].set_xlabel('Boiler pressure (MPa)'); ax[0].set_ylabel('Efficiency (%)'); ax[0].set_title('↑ boiler pressure')
Tm = np.linspace(400, 600, 11) + 273.15
ax[1].plot(Tm-273.15, [rankine(P_b, t, P_c)['eff']*100 for t in Tm], color=CYAN, marker='o')
ax[1].set_xlabel('Turbine inlet T (°C)'); ax[1].set_title('↑ superheat')
Pc = np.array([5,10,20,50,100])*1e3
ax[2].plot(Pc/1e3, [rankine(P_b, T3, p)['eff']*100 for p in Pc], color=CYAN, marker='s')
ax[2].set_xlabel('Condenser pressure (kPa)'); ax[2].set_title('↓ condenser pressure'); ax[2].invert_xaxis()
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 4. Trade-off — the wetness limit
Higher boiler pressure and lower condenser pressure both push the turbine exit (state 4) **wetter**. Below ~88 %
quality, water droplets erode the last turbine stages — which is exactly why real plants use **reheat**.

In [ ]:
for p in [4e6, 8e6, 12e6, 16e6]:
    r = rankine(p, T3, P_c)
    print(f'Boiler {p/1e6:>4.0f} MPa  ->  eff {r["eff"]*100:4.1f} %   exit quality {r["x4"]*100:4.1f} %')

## 5. Conclusion

- The 8 MPa / 500 °C / 10 kPa plant reaches **≈ 39 % ideal** and **≈ 33 % real** thermal efficiency; the back-work ratio is under 1 % (the pump is almost free — the signature advantage of a vapour cycle over a gas cycle).
- All three classic levers improve efficiency, but raising boiler pressure and lowering condenser pressure drive the **turbine exit wetter** — the real design constraint.
- The natural next step is **reheat** (expand, return to the boiler, expand again): it lifts efficiency *and* keeps the exit dry. A regenerative feedwater heater would add a further gain.

*This is the analysis that sizes a plant: it fixes the operating point, quantifies efficiency, and shows where the design hits its physical limit.*